# IDRiD Diabetic Retinopathy Severity Classification
### EfficientNet-B3 · Transfer Learning · Bayesian HPO

**Dataset:** IDRiD — Ghulamay ALi1
**Task:** Classify retinal fundus images into 5 DR severity grades (0 = No DR → 4 = Proliferative DR).

## 1. Imports & Setup

In [ ]:
import os, sys, random, logging
from datetime import datetime
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, Optional, Tuple, Union, Callable

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    cohen_kappa_score, roc_auc_score, f1_score,
    classification_report, confusion_matrix,
)

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

NUM_CLASSES = 5   # DR grades 0-4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True

print(f"Device : {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")


## 2. Logging

In [ ]:
_LOG_FORMAT = "[%(levelname)-8s] [%(name)s] %(message)s"

def setup_logging(log_dir=None, level=logging.INFO, log_to_file=True):
    root = logging.getLogger()
    for h in root.handlers[:]: root.removeHandler(h)
    root.setLevel(logging.DEBUG)
    
    # 2. Remove datefmt as it is no longer needed
    fmt = logging.Formatter(_LOG_FORMAT)

    ch = logging.StreamHandler(sys.stdout)
    ch.setFormatter(fmt); ch.setLevel(level)
    root.addHandler(ch)

    log_file = None
    if log_to_file and log_dir is not None:
        log_dir = Path(log_dir); log_dir.mkdir(parents=True, exist_ok=True)
        log_file = log_dir / f"run_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"
        fh = logging.FileHandler(log_file, encoding="utf-8")
        fh.setFormatter(fmt); fh.setLevel(logging.DEBUG)
        root.addHandler(fh)

    root.info(f"Logging ready (console={logging.getLevelName(level)}, file={log_file or 'off'})")
    return log_file

def get_logger(name): return logging.getLogger(name)

## 3. Path Discovery

In [ ]:
KAGGLE_INPUT   = Path("/kaggle/input")
KAGGLE_WORKING = Path("/kaggle/working")

def is_kaggle(): return KAGGLE_INPUT.exists() or os.environ.get("KAGGLE_KERNEL_RUN_TYPE") is not None

def get_output_root():
    r = KAGGLE_WORKING if is_kaggle() else Path("./outputs")
    r.mkdir(parents=True, exist_ok=True); return r

def get_log_dir():
    d = get_output_root() / "logs"; d.mkdir(parents=True, exist_ok=True); return d

def _find_dir(root, *kws):
    if not root.exists(): return None
    kws = [k.lower() for k in kws]
    cands = [p for p in root.rglob("*") if p.is_dir() and all(k in p.name.lower() for k in kws)]
    return min(cands, key=lambda p: len(p.parts)) if cands else None

def _find_file(root, *kws, suffix=".csv"):
    if not root.exists(): return None
    kws = [k.lower() for k in kws]
    cands = [p for p in root.rglob(f"*{suffix}") if all(k in p.name.lower() for k in kws)]
    return min(cands, key=lambda p: len(p.parts)) if cands else None

def _tree(root, depth=4, pre=""):
    lines = []
    if not root.exists() or depth < 0: return lines
    try: children = sorted(p for p in root.iterdir() if p.is_dir())
    except PermissionError: return [f"{pre}{root.name}/ (no access)"]
    for c in children:
        lines.append(f"{pre}{c.name}/")
        if depth > 0: lines.extend(_tree(c, depth-1, pre+"  "))
    return lines

def find_idrid_root(local="./data/idrid"):
    log = get_logger(__name__)
    if is_kaggle():
        for kws in (("disease","grading"), ("idrid",), ("diabetic","retinopathy")):
            m = _find_dir(KAGGLE_INPUT, *kws)
            if m is not None:
                root = m.parent if kws == ("disease","grading") else m
                log.info(f"IDRiD found at: {root}"); return root
        tree = "\n".join(_tree(KAGGLE_INPUT, 4)) or "(empty)"
        raise FileNotFoundError(
            f"IDRiD not found under /kaggle/input.\nTree:\n{tree}\n\n"
            "Override: IDRiDPaths.discover(root=Path('/kaggle/input/...'))"
        )
    p = Path(local)
    if not p.exists(): log.warning(f"{p} missing -- set local= correctly")
    return p

@dataclass
class IDRiDPaths:
    root: Path
    train_images_dir: Optional[Path] = None
    test_images_dir:  Optional[Path] = None
    train_labels_csv: Optional[Path] = None
    test_labels_csv:  Optional[Path] = None

    @classmethod
    def discover(cls, root=None):
        root = root or find_idrid_root()
        gr   = _find_dir(root, "disease","grading") or root
        p = cls(
            root=root,
            train_images_dir = _find_dir(gr, "train"),
            test_images_dir  = _find_dir(gr, "test"),
            train_labels_csv = _find_file(gr, "train","label") or _find_file(gr, "train"),
            test_labels_csv  = _find_file(gr, "test","label")  or _find_file(gr, "test"),
        )
        p._log()
        return p

    def _log(self):
        log = get_logger(__name__)
        log.info("IDRiD paths:")
        for k in ("root","train_images_dir","test_images_dir","train_labels_csv","test_labels_csv"):
            log.info(f"  {k:20s}: {getattr(self,k)}")
        missing = [k for k in ("train_images_dir","test_images_dir","train_labels_csv","test_labels_csv") if not getattr(self,k)]
        if missing: log.warning(f"Could not auto-locate: {missing} -- set manually if needed")


## 4. Data Loading & Split

In [ ]:
def _col(df, *kws):
    kws = [k.lower() for k in kws]
    for c in df.columns:
        if all(k in c.lower().replace("_"," ").strip() for k in kws): return c
    raise KeyError(f"No column matching {kws} in {list(df.columns)}")

def load_idrid_labels(csv_path):
    log = get_logger(__name__)
    df = pd.read_csv(csv_path)
    df = df[[_col(df,"image"), _col(df,"retinopathy")]].copy()
    df.columns = ["image_name","dr_grade"]
    df["image_name"] = df["image_name"].astype(str).str.strip()
    df = df[df["image_name"]!=""].dropna().copy()
    df["dr_grade"] = df["dr_grade"].astype(int)
    df = df.reset_index(drop=True)
    log.info(f"Loaded {len(df)} images from {csv_path}")
    log.info(f"Class distribution:\n{df['dr_grade'].value_counts().sort_index().to_string()}")
    return df

setup_logging(log_dir=get_log_dir())
logger = get_logger(__name__)

paths = IDRiDPaths.discover()

train_df = load_idrid_labels(paths.train_labels_csv)
test_df  = load_idrid_labels(paths.test_labels_csv)

train_split_df, val_split_df = train_test_split(
    train_df, test_size=0.15, stratify=train_df["dr_grade"], random_state=SEED
)
print(f"Train: {len(train_split_df)} | Val: {len(val_split_df)} | Test: {len(test_df)}")
train_split_df["dr_grade"].value_counts().sort_index()


## 5. Preprocessing

In [ ]:
def load_image_rgb(path):
    img = cv2.imread(path, cv2.IMREAD_COLOR)
    if img is None: raise FileNotFoundError(f"Cannot read: {path}")
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

def remove_black_borders(img, tol=7):
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    mask = gray > tol
    rows, cols = np.any(mask, axis=1), np.any(mask, axis=0)
    if not rows.any() or not cols.any(): return img
    y0,y1 = np.where(rows)[0][[0,-1]]; x0,x1 = np.where(cols)[0][[0,-1]]
    c = img[y0:y1+1, x0:x1+1]
    return c if c.shape[0]>=10 and c.shape[1]>=10 else img

def resize_image(img, target_size=300):
    interp = cv2.INTER_AREA if img.shape[0] > target_size else cv2.INTER_LANCZOS4
    return cv2.resize(img, (target_size, target_size), interpolation=interp)

def fill_black_corners(img, tol=7, inpaint_radius=5):
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    bm = (gray <= tol).astype(np.uint8) * 255
    return cv2.inpaint(img, bm, inpaint_radius, cv2.INPAINT_TELEA) if bm.sum() else img

def apply_clahe(img, clip_limit=2.0, tile_grid_size=(8,8)):
    lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
    l,a,b = cv2.split(lab)
    l = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size).apply(l)
    return cv2.cvtColor(cv2.merge((l,a,b)), cv2.COLOR_LAB2RGB)

def ben_graham_preprocessing(img, sigma_fraction=1/30):
    sigma = max(img.shape[1]*sigma_fraction, 1.0)
    return cv2.addWeighted(img, 4, cv2.GaussianBlur(img,(0,0),sigmaX=sigma), -4, 128)

def preprocess_fundus_image(img, target_size=300, use_clahe=True, use_ben_graham=True,
                             clahe_clip_limit=2.0, clahe_tile_grid_size=(8,8),
                             ben_graham_sigma_fraction=1/30):
    img = remove_black_borders(img)
    img = resize_image(img, target_size)
    img = fill_black_corners(img)
    if use_clahe:       img = apply_clahe(img, clahe_clip_limit, clahe_tile_grid_size)
    if use_ben_graham:  img = ben_graham_preprocessing(img, ben_graham_sigma_fraction)
    return img


In [ ]:
_row = train_split_df.iloc[0]
_raw = load_image_rgb(str(paths.train_images_dir / f"{_row['image_name']}.jpg"))
_proc = preprocess_fundus_image(_raw)
fig, axes = plt.subplots(1,2,figsize=(10,5))
axes[0].imshow(_raw);  axes[0].set_title(f"Raw (grade {_row['dr_grade']})"); axes[0].axis("off")
axes[1].imshow(_proc); axes[1].set_title("Preprocessed");                   axes[1].axis("off")
plt.tight_layout(); plt.show()

## 6. Dataset & DataLoaders

In [ ]:
class IDRiDDataset(Dataset):
    def __init__(self, df, image_dir, target_size=300, use_clahe=True,
                 use_ben_graham=True, transform=None, image_extension=".jpg"):
        self.df = df.reset_index(drop=True)
        self.image_dir = Path(image_dir)
        self.target_size = target_size
        self.use_clahe = use_clahe
        self.use_ben_graham = use_ben_graham
        self.transform = transform
        self.image_extension = image_extension

    def __len__(self): return len(self.df)

    def _path(self, name):
        p = self.image_dir / f"{name}{self.image_extension}"
        if p.exists(): return p
        for ext in (".jpg",".jpeg",".png",".tif",".tiff"):
            alt = self.image_dir / f"{name}{ext}"
            if alt.exists(): return alt
        return p

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = load_image_rgb(str(self._path(row["image_name"])))
        img = preprocess_fundus_image(img, self.target_size, self.use_clahe, self.use_ben_graham)
        if self.transform is not None:
            img = self.transform(img)
        else:
            img = torch.from_numpy(img.transpose(2,0,1).copy()).float() / 255.0
        return img, torch.tensor(int(row["dr_grade"]), dtype=torch.long)


TARGET_SIZE  = 300
BATCH_SIZE   = 16
NUM_WORKERS  = 2

train_dataset = IDRiDDataset(train_split_df, paths.train_images_dir, target_size=TARGET_SIZE)
val_dataset   = IDRiDDataset(val_split_df,   paths.train_images_dir, target_size=TARGET_SIZE)
test_dataset  = IDRiDDataset(test_df,        paths.test_images_dir,  target_size=TARGET_SIZE)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                           num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                           num_workers=NUM_WORKERS, pin_memory=True)

_imgs, _labels = next(iter(train_loader))
print(f"Batch shape: {tuple(_imgs.shape)} | sample labels: {_labels.tolist()}")


## 7. Model — EfficientNet-B3

Backbone: EfficientNet-B3 (ImageNet-pretrained, 10.7M params).

**Two-phase fine-tuning:**
- Phase 1: freeze `model.features` (backbone), train head only → 7,685 params.
- Phase 2: partial unfreeze `features[unfreeze_from_stage:]`

In [ ]:
from torchvision import models
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.tensorboard import SummaryWriter

def build_efficientnet_b3(num_classes=NUM_CLASSES, pretrained=True, dropout=0.3):
    weights = models.EfficientNet_B3_Weights.IMAGENET1K_V1 if pretrained else None
    m = models.efficientnet_b3(weights=weights, dropout=dropout)
    m.classifier[1] = nn.Linear(m.classifier[1].in_features, num_classes)
    return m

def set_backbone_trainable(model, trainable=True, unfreeze_from_stage=None):
    if not trainable:
        for p in model.features.parameters(): p.requires_grad = False
    elif unfreeze_from_stage is None:
        for p in model.features.parameters(): p.requires_grad = True
    else:
        for i, stage in enumerate(model.features.children()):
            for p in stage.parameters(): p.requires_grad = (i >= unfreeze_from_stage)
    for p in model.classifier.parameters(): p.requires_grad = True

def count_params(model, trainable_only=False):
    return sum(p.numel() for p in model.parameters()
               if (p.requires_grad or not trainable_only))

_test_model = build_efficientnet_b3(pretrained=True).to(DEVICE)
set_backbone_trainable(_test_model, trainable=False)
print(f"EfficientNet-B3 on {DEVICE}")
print(f"Total params    : {count_params(_test_model):,}")
print(f"Phase 1 trainable: {count_params(_test_model, trainable_only=True):,}")

_test_model.eval()
with torch.no_grad():
    _out = _test_model(_imgs.to(DEVICE))
assert _out.shape == (_imgs.shape[0], NUM_CLASSES)
print(f"Forward pass OK: {tuple(_imgs.shape)} -> {tuple(_out.shape)}")
del _test_model

## 8. Loss Function — Focal Loss

In [ ]:
def compute_class_weights(df, num_classes=NUM_CLASSES):
    counts = df["dr_grade"].value_counts().reindex(range(num_classes), fill_value=0).clip(lower=1)
    weights = counts.sum() / (num_classes * counts)
    return torch.tensor(weights.values, dtype=torch.float32)

class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, reduction="mean"):
        super().__init__()
        self.alpha = alpha; self.gamma = gamma; self.reduction = reduction

    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, reduction="none")
        loss = ((1 - torch.exp(-ce)) ** self.gamma) * ce
        if self.alpha is not None:
            a = self.alpha.to(logits.device)[targets]
            loss = a * loss
            if self.reduction == "mean": return loss.sum() / a.sum()
        return loss.mean() if self.reduction == "mean" else (loss.sum() if self.reduction == "sum" else loss)

class_weights = compute_class_weights(train_split_df).to(DEVICE)
print(f"Class weights: {[round(w,3) for w in class_weights.tolist()]}")

criterion = FocalLoss(alpha=class_weights, gamma=2.0)
print("Loss: FocalLoss (gamma=2.0, alpha=class_weights)")

## 9. Training Utilities

In [ ]:
import shutil
CKPT_SAVE_PATH = Path("/kaggle/working/checkpoints/best_model_FINAL.pt")
CKPT_SAVE_PATH.parent.mkdir(parents=True, exist_ok=True)

CHECKPOINT_DIR = get_output_root() / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

writer = SummaryWriter(log_dir=str(get_output_root() / "tensorboard_logs"))

class EarlyStopping:
    def __init__(self, patience=5, min_delta=1e-4):
        self.patience = patience; self.min_delta = min_delta
        self.best_loss = float("inf"); self.counter = 0; self.should_stop = False

    def step(self, val_loss):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss; self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience: self.should_stop = True

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total, n = 0.0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(imgs), labels)
        loss.backward(); optimizer.step()
        total += loss.item() * imgs.size(0); n += imgs.size(0)
    return total / n

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total, correct, n = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        out = model(imgs)
        total += criterion(out, labels).item() * imgs.size(0)
        correct += (out.argmax(dim=1) == labels).sum().item()
        n += imgs.size(0)
    return total/n, correct/n

def save_checkpoint(model, optimizer, epoch, val_loss, path):
    torch.save({"epoch": epoch, "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(), "val_loss": val_loss}, path)

def run_phase(model, phase_name, num_epochs, optimizer, epoch_offset,
              best_val_loss, ckpt_path, early_stop_patience=5):
    scheduler = ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2)
    stopper   = EarlyStopping(patience=early_stop_patience)
    last_epoch = epoch_offset

    print(f"=== {phase_name}: {num_epochs} epochs ===")
    for epoch in range(1, num_epochs+1):
        train_loss = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE)
        val_loss, val_acc = evaluate(model, val_loader, criterion, DEVICE)
        scheduler.step(val_loss)

        g = epoch_offset + epoch; last_epoch = g
        print(f"[{phase_name}][Epoch {epoch}/{num_epochs}] "
                    f"train={train_loss:.4f} val={val_loss:.4f} acc={val_acc:.4f}")

        writer.add_scalars("loss", {"train": train_loss, "val": val_loss}, g)
        writer.add_scalar("val_acc", val_acc, g)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            save_checkpoint(model, optimizer, g, val_loss, ckpt_path)
            print(f"  -> best val_loss={val_loss:.4f}, saved")

        stopper.step(val_loss)
        if stopper.should_stop:
            print(f"  -> early stopping at epoch {epoch}"); break

    return best_val_loss, last_epoch


## 10. Hyperparameter Optimization (Bayesian — Optuna TPE)

Search space: Phase 1/2 LRs, dropout, weight decay, unfreeze stage.
Each trial runs a reduced budget (15+15 epochs with early stopping) as a
cheap proxy for ranking configurations..

In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

HPO_PHASE1_EPOCHS = 15
HPO_PHASE2_EPOCHS = 15
N_TRIALS          = 7  
EARLY_STOP_PATIENCE = 5

def hpo_objective(trial):
    phase1_lr   = trial.suggest_float("phase1_lr",          1e-4, 1e-2,  log=True)
    p2_lr_head  = trial.suggest_float("phase2_lr_head",     1e-5, 1e-3,  log=True)
    p2_lr_back  = trial.suggest_float("phase2_lr_backbone", 1e-6, 1e-4,  log=True)
    dropout     = trial.suggest_float("dropout",            0.2,  0.5)
    weight_decay= trial.suggest_float("weight_decay",       1e-5, 1e-3,  log=True)
    stage       = trial.suggest_categorical("unfreeze_from_stage", [6, 7, 8])

    m     = build_efficientnet_b3(pretrained=True, dropout=dropout).to(DEVICE)
    ckpt  = CHECKPOINT_DIR / f"hpo_trial_{trial.number}.pt"
    best  = float("inf")

    # Phase 1
    set_backbone_trainable(m, trainable=False)
    opt = optim.AdamW(filter(lambda p: p.requires_grad, m.parameters()),
                      lr=phase1_lr, weight_decay=weight_decay)
    best, last = run_phase(m, f"Trial {trial.number} P1",
                           HPO_PHASE1_EPOCHS, opt, 0, best, ckpt, EARLY_STOP_PATIENCE)

    m.load_state_dict(torch.load(ckpt, map_location=DEVICE, weights_only=False)["model_state_dict"])

    # Phase 2
    set_backbone_trainable(m, trainable=True, unfreeze_from_stage=stage)
    bb = [p for p in m.features.parameters() if p.requires_grad]
    opt = optim.AdamW([{"params": bb, "lr": p2_lr_back},
                       {"params": m.classifier.parameters(), "lr": p2_lr_head}],
                      weight_decay=weight_decay)
    best, _ = run_phase(m, f"Trial {trial.number} P2",
                        HPO_PHASE2_EPOCHS, opt, last, best, ckpt, EARLY_STOP_PATIENCE)

    # Score by QWK
    m.load_state_dict(torch.load(ckpt, map_location=DEVICE, weights_only=False)["model_state_dict"])
    m.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for imgs, labels in val_loader:
            y_pred.extend(m(imgs.to(DEVICE)).argmax(dim=1).cpu().numpy())
            y_true.extend(labels.numpy())
    qwk = cohen_kappa_score(y_true, y_pred, weights="quadratic")
    print(f"Trial {trial.number} QWK={qwk:.4f}")

    ckpt.unlink(missing_ok=True)   # don't fill disk with trial weights
    return qwk

study = optuna.create_study(direction="maximize",
                             sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(hpo_objective, n_trials=N_TRIALS)

print(f"HPO complete. Best QWK: {study.best_value:.4f}")
print(f"Best params: {study.best_params}")


## 11. Final Training (HPO best params)

In [ ]:
best = study.best_params

PHASE1_EPOCHS = 50
PHASE2_EPOCHS = 60
best_ckpt_path = CHECKPOINT_DIR / "best_model.pt"

model = build_efficientnet_b3(pretrained=True, dropout=best["dropout"]).to(DEVICE)
best_val_loss = float("inf")

# Phase 1: frozen backbone, head only
set_backbone_trainable(model, trainable=False)
optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                         lr=best["phase1_lr"], weight_decay=best["weight_decay"])
best_val_loss, last_epoch = run_phase(
    model, "Final Phase 1", PHASE1_EPOCHS, optimizer, 0, best_val_loss, best_ckpt_path
)

p1 = torch.load(best_ckpt_path, map_location=DEVICE, weights_only=False)
model.load_state_dict(p1["model_state_dict"])
print(f"Restored Phase 1 best: epoch {p1['epoch']}, val_loss={p1['val_loss']:.4f}")

set_backbone_trainable(model, trainable=True,
                        unfreeze_from_stage=best["unfreeze_from_stage"])
bb = [p for p in model.features.parameters() if p.requires_grad]
optimizer = optim.AdamW([{"params": bb,                            "lr": best["phase2_lr_backbone"]},
                          {"params": model.classifier.parameters(), "lr": best["phase2_lr_head"]}],
                         weight_decay=best["weight_decay"])
print(f"[Phase 2] Trainable: {count_params(model, trainable_only=True):,} / {count_params(model):,}")
best_val_loss, _ = run_phase(
    model, "Final Phase 2", PHASE2_EPOCHS, optimizer, last_epoch, best_val_loss, best_ckpt_path
)

writer.close()
print(f"Training complete. Best val_loss={best_val_loss:.4f} | ckpt={best_ckpt_path}")


In [ ]:
import os
from datetime import datetime

size = os.path.getsize(best_ckpt_path) / 1e6
print(f"Checkpoint saved: {best_ckpt_path} ({size:.1f} MB)")

backup = Path(f"/kaggle/working/checkpoints/effnetb3_backup_{datetime.now().strftime('%H%M%S')}.pt")
shutil.copy(best_ckpt_path, backup)
print(f"Backup saved: {backup}")

## 12. Evaluation

In [ ]:
@torch.no_grad()
def get_predictions(model, loader, device):
    model.eval()
    yt, yp, ypr = [], [], []
    for imgs, labels in loader:
        probs = F.softmax(model(imgs.to(device)), dim=1).cpu().numpy()
        yt.extend(labels.numpy()); yp.extend(probs.argmax(axis=1)); ypr.extend(probs)
    return np.array(yt), np.array(yp), np.array(ypr)

def evaluate_full(model, loader, device, num_classes=NUM_CLASSES, split_name="val"):
    y_true, y_pred, y_probs = get_predictions(model, loader, device)
    qwk      = cohen_kappa_score(y_true, y_pred, weights="quadratic")
    macro_f1 = f1_score(y_true, y_pred, average="macro")
    try:    auc = roc_auc_score(y_true, y_probs, multi_class="ovr", average="macro")
    except: auc = float("nan"); logger.warning("AUC-ROC not computable (class missing from split)")
    cm = confusion_matrix(y_true, y_pred, labels=list(range(num_classes)))
    report = classification_report(y_true, y_pred, zero_division=0,
                                    target_names=[f"Grade {i}" for i in range(num_classes)])
    print(f"=== {split_name} ===")
    print(f"QWK={qwk:.4f}  Macro F1={macro_f1:.4f}  AUC-ROC={auc:.4f}")
    print(f"Per-class report:\n{report}")
    return {"qwk":qwk,"macro_f1":macro_f1,"auc":auc,"confusion_matrix":cm,
            "y_true":y_true,"y_pred":y_pred,"y_probs":y_probs}

def plot_confusion_matrix(cm, num_classes=NUM_CLASSES, title="Confusion Matrix"):
    fig, ax = plt.subplots(figsize=(6,5))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(num_classes)); ax.set_xticklabels([f"Grade {i}" for i in range(num_classes)])
    ax.set_yticks(range(num_classes)); ax.set_yticklabels([f"Grade {i}" for i in range(num_classes)])
    ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(title)
    for i in range(num_classes):
        for j in range(num_classes):
            ax.text(j,i,cm[i,j],ha="center",va="center",
                    color="white" if cm[i,j]>cm.max()/2 else "black")
    plt.colorbar(im); plt.tight_layout(); plt.show()


In [ ]:
best_ckpt = torch.load(best_ckpt_path, map_location=DEVICE, weights_only=False)
model.load_state_dict(best_ckpt["model_state_dict"])
print(f"Loaded checkpoint: epoch {best_ckpt['epoch']}, val_loss={best_ckpt['val_loss']:.4f}")

val_metrics = evaluate_full(model, val_loader, DEVICE, split_name="validation")
plot_confusion_matrix(val_metrics["confusion_matrix"], title="Validation Confusion Matrix")


## 13. Analysis — Grad-CAM & Error Distances

In [ ]:
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model; self.activations = None; self.gradients = None
        self._fh = target_layer.register_forward_hook(
            lambda m,i,o: setattr(self,"activations",o.detach()))
        self._bh = target_layer.register_full_backward_hook(
            lambda m,gi,go: setattr(self,"gradients",go[0].detach()))

    def generate(self, inp, target_class=None):
        self.model.eval()
        x = inp.clone().requires_grad_(True)
        out = self.model(x)
        tc  = out.argmax(dim=1).item() if target_class is None else target_class
        self.model.zero_grad(); out[0,tc].backward()
        w   = self.gradients.mean(dim=(2,3),keepdim=True)
        cam = F.relu((w*self.activations).sum(dim=1)).squeeze().cpu().numpy()
        if cam.max()>0: cam = cam/cam.max()
        h,w2 = inp.shape[2], inp.shape[3]
        return cv2.resize(cam,(w2,h)), tc

    def close(self): self._fh.remove(); self._bh.remove()

def overlay_gradcam(image, cam, alpha=0.4):
    hm = cv2.cvtColor(cv2.applyColorMap(np.uint8(255*cam), cv2.COLORMAP_JET), cv2.COLOR_BGR2RGB)
    return (alpha*hm + (1-alpha)*image).astype(np.uint8)

def show_gradcam_for_examples(model, dataset, indices, device, target_layer, title_prefix=""):
    if not len(indices): print(f"{title_prefix}: no examples."); return
    gc  = GradCAM(model, target_layer)
    n   = len(indices)
    fig, axes = plt.subplots(n,2,figsize=(8,4*n))
    if n==1: axes = axes.reshape(1,2)
    for row,idx in enumerate(indices):
        t,label = dataset[idx]
        cam,pred = gc.generate(t.unsqueeze(0).to(device))
        img_np = (t.permute(1,2,0).cpu().numpy()*255).astype(np.uint8)
        axes[row,0].imshow(img_np)
        axes[row,0].set_title(f"{title_prefix} idx={idx}  True=Grade{label.item()}"); axes[row,0].axis("off")
        axes[row,1].imshow(overlay_gradcam(img_np,cam))
        axes[row,1].set_title(f"Grad-CAM  Pred=Grade{pred}"); axes[row,1].axis("off")
    plt.tight_layout(); plt.show(); gc.close()

def analyze_error_distances(y_true, y_pred, num_classes=NUM_CLASSES):
    d = np.abs(y_true-y_pred); total = len(y_true)
    print(f"=== Error distance analysis ({total} examples) ===")
    print(f"Correct (d=0): {(d==0).sum()} ({(d==0).mean()*100:.1f}%)")
    for i in range(1,num_classes):
        n=(d==i).sum()
        if n: print(f"d={i}: {n} ({n/total*100:.1f}%)")
    sv=(d>=3).sum(); print(f"Severe (d>=3): {sv} ({sv/total*100:.1f}%)")
    return d

def plot_error_histogram(d, title="Error Distance Distribution"):
    fig,ax = plt.subplots(figsize=(6,4))
    md = int(d.max())
    ax.bar(range(md+1),[np.sum(d==i) for i in range(md+1)],
           color=["#2ca02c"]+["#ff7f0e"]*md)
    ax.set_xlabel("|true - predicted|"); ax.set_ylabel("Count")
    ax.set_title(title); ax.set_xticks(range(md+1)); plt.tight_layout(); plt.show()


In [ ]:
tl = model.features[8]

yt, yp = val_metrics["y_true"], val_metrics["y_pred"]
g1_miss      = np.where((yt==1)&(yp!=1))[0]
g3_to_0_miss = np.where((yt==3)&(yp==0))[0]
print(f"Grade 1 misses      : {g1_miss.tolist()}")
print(f"Grade 3->0 misses   : {g3_to_0_miss.tolist()}")

show_gradcam_for_examples(model, val_dataset, g1_miss,      DEVICE, tl, "Grade1 miss")
show_gradcam_for_examples(model, val_dataset, g3_to_0_miss, DEVICE, tl, "Grade3->0 miss")

rng = np.random.default_rng(0)
for cls in range(NUM_CLASSES):
    idx = np.where(yt==cls)[0]
    if not len(idx): continue
    show_gradcam_for_examples(model, val_dataset,
        rng.choice(idx,size=min(2,len(idx)),replace=False).tolist(),
        DEVICE, tl, f"True=Grade{cls}")

# Error distances
val_d = analyze_error_distances(yt, yp)
plot_error_histogram(val_d, "Validation Error Distance Distribution")


## 14. Test Set Evaluation

In [ ]:
test_metrics = evaluate_full(model, test_loader, DEVICE, split_name="TEST (final, one-time)")
plot_confusion_matrix(test_metrics["confusion_matrix"], title="Test Set Confusion Matrix")

test_d = analyze_error_distances(test_metrics["y_true"], test_metrics["y_pred"])
plot_error_histogram(test_d, "Test Set Error Distance Distribution")

print("=== FINAL REPORTED TEST RESULTS ===")
print(f"QWK={test_metrics['qwk']:.4f}  "
            f"Macro F1={test_metrics['macro_f1']:.4f}  "
            f"AUC-ROC={test_metrics['auc']:.4f}")

In [ ]:
import os, json, shutil
from pathlib import Path

OUTPUT_DIR = Path("/kaggle/working/results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ---- 1. Metrics ----
with open(OUTPUT_DIR / "effnet_metrics.json", "w") as f:
    json.dump({
        "val":  {"qwk": float(val_metrics["qwk"]),  "f1": float(val_metrics["macro_f1"]),  "auc": float(val_metrics["auc"])},
        "test": {"qwk": float(test_metrics["qwk"]), "f1": float(test_metrics["macro_f1"]), "auc": float(test_metrics["auc"])},
    }, f, indent=2)
logger.info("Saved effnet_metrics.json")

# ---- 2. Confusion matrices ----
for split_name, m in [("validation", val_metrics), ("test", test_metrics)]:
    cm = m["confusion_matrix"]
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(NUM_CLASSES)); ax.set_xticklabels([f"Grade {i}" for i in range(NUM_CLASSES)])
    ax.set_yticks(range(NUM_CLASSES)); ax.set_yticklabels([f"Grade {i}" for i in range(NUM_CLASSES)])
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    ax.set_title(f"EfficientNet-B3 — {split_name}\nQWK={m['qwk']:.4f}  F1={m['macro_f1']:.4f}")
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            ax.text(j, i, cm[i, j], ha="center", va="center",
                    color="white" if cm[i, j] > cm.max() / 2 else "black")
    plt.colorbar(im); plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"cm_effnet_{split_name}.png", dpi=150, bbox_inches="tight")
    plt.close()
    logger.info(f"Saved cm_effnet_{split_name}.png")

# ---- 3. Prediction scatter (test) ----
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(test_metrics["y_true"], test_metrics["y_pred"], alpha=0.6, s=40, color="steelblue")
ax.plot([-0.3, 4.3], [-0.3, 4.3], "r--", linewidth=1, label="Perfect prediction")
ax.set_xlim(-0.3, 4.3); ax.set_ylim(-0.3, 4.3)
ax.set_xlabel("True Grade"); ax.set_ylabel("Predicted Grade")
ax.set_title(f"EfficientNet-B3 — Test Predictions\nQWK={test_metrics['qwk']:.4f}")
ax.set_xticks(range(5)); ax.set_yticks(range(5))
ax.legend(); ax.grid(True, alpha=0.3); plt.tight_layout()
plt.savefig(OUTPUT_DIR / "scatter_effnet_test.png", dpi=150, bbox_inches="tight")
plt.close()
logger.info("Saved scatter_effnet_test.png")

try:
    gradcam_dir = OUTPUT_DIR / "gradcam"
    gradcam_dir.mkdir(exist_ok=True)
    target_layer = model.features[8]
    gc = GradCAM(model, target_layer)
    y_true = test_metrics["y_true"]
    y_pred = test_metrics["y_pred"]
    rng = np.random.default_rng(42)
    for cls in range(NUM_CLASSES):
        idx = np.where(y_true == cls)[0]
        if len(idx) == 0: continue
        for i, sample_idx in enumerate(rng.choice(idx, size=min(2, len(idx)), replace=False)):
            img_tensor, label = test_dataset[sample_idx]
            cam, pred = gc.generate(img_tensor.unsqueeze(0).to(DEVICE))
            img_np = (img_tensor.permute(1,2,0).cpu().numpy() * 255).astype(np.uint8)
            overlay = overlay_gradcam(img_np, cam)
            fig, axes = plt.subplots(1, 2, figsize=(8, 4))
            axes[0].imshow(img_np);  axes[0].set_title(f"True=Grade{label.item()}"); axes[0].axis("off")
            axes[1].imshow(overlay); axes[1].set_title(f"Grad-CAM  Pred=Grade{pred}"); axes[1].axis("off")
            plt.tight_layout()
            plt.savefig(gradcam_dir / f"gradcam_grade{cls}_sample{i}.png", dpi=150, bbox_inches="tight")
            plt.close()
    gc.close()
    logger.info(f"Grad-CAM images saved to {gradcam_dir}")
except Exception as e:
    logger.warning(f"Grad-CAM skipped: {e}")


test_distances = np.abs(test_metrics["y_true"] - test_metrics["y_pred"])

fig, ax = plt.subplots(figsize=(6, 4))
max_d = int(test_distances.max())
counts = [int(np.sum(test_distances == d)) for d in range(max_d + 1)]
ax.bar(range(max_d + 1), counts, color=["#2ca02c"] + ["#ff7f0e"] * max_d)
ax.set_xlabel("|True grade - Predicted grade|")
ax.set_ylabel("Count"); ax.set_title("Test Set Error Distance Distribution")
ax.set_xticks(range(max_d + 1)); ax.grid(True, alpha=0.3); plt.tight_layout()
plt.savefig(OUTPUT_DIR / "error_distances_test.png", dpi=150, bbox_inches="tight")
plt.close()
logger.info("Saved error_distances_test.png")

# ---- 6. Copy checkpoint ----
ckpt_dir = OUTPUT_DIR / "checkpoints"
ckpt_dir.mkdir(exist_ok=True)
shutil.copy(best_ckpt_path, ckpt_dir / "best_model_FINAL.pt")
logger.info("Checkpoint copied")

# ---- Summary ----
print("\n" + "="*55)
print("ALL OUTPUTS SAVED TO /kaggle/working/results/")
print("="*55)
for f in sorted(OUTPUT_DIR.rglob("*")):
    if f.is_file():
        print(f"  {str(f.relative_to(OUTPUT_DIR)):45s} {f.stat().st_size/1024:.1f} KB")